In [31]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [32]:
training_set = tf.keras.preprocessing.image_dataset_from_directory(
    "Crop Diseases",
    labels="inferred",
    label_mode="categorical",
    class_names= None,
    color_mode="rgb",
    batch_size=64,
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="training",
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 13324 files belonging to 17 classes.
Using 10660 files for training.


In [33]:
validation_set = tf.keras.preprocessing.image_dataset_from_directory(
    "Crop Diseases",
    labels="inferred",
    label_mode="categorical",
    class_names= None,
    color_mode="rgb",
    batch_size=64,
    image_size=(128, 128),
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="validation",
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

Found 13324 files belonging to 17 classes.
Using 2664 files for validation.


In [34]:
AUTOTUNE = tf.data.AUTOTUNE
training_set = training_set.cache().prefetch(buffer_size=AUTOTUNE)
validation_set = validation_set.cache().prefetch(buffer_size=AUTOTUNE)

In [35]:
import ssl
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [36]:

ssl._create_default_https_context = ssl._create_unverified_context

# 1. Base model
base_model = MobileNetV2(
    input_shape=(128, 128, 3),
    include_top=False,
    weights='imagenet'
)

# 2. Pipeline with explicit MobileNet scaling [-1, 1]
inputs = tf.keras.Input(shape=(128, 128, 3))

x = layers.RandomFlip("horizontal_and_vertical")(inputs)
x = layers.RandomRotation(0.15)(x)

x = preprocess_input(inputs) 
x = base_model(x, training=False)  # Locks BatchNormalization in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(17, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

# 3. Unfreeze top 30 layers
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

# 4. Compile with fine-tuning learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [37]:
callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=0.000001, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
]

In [ ]:

# 5. Fit model
history = model.fit(
    x=training_set,
    validation_data=validation_set,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
167/167 ━━━━━━━━━━━━━━━━━━━━ 29s 137ms/step - accuracy: 0.8101 - loss: 0.5801 - val_accuracy: 0.7909 - val_loss: 0.6980 - learning_rate: 1.0000e-04
Epoch 2/25
152/167 ━━━━━━━━━━━━━━━━━━━━ 1s 75ms/step - accuracy: 0.9289 - loss: 0.2000